# 01 · SFT через LoRA

**Правило:** содержательные решения принимает студент. Нет данных — уточнить, есть данные — дать варианты, справочный вопрос — ответить.

**Схема:** замер до → навесить адаптер → обучить → замер после на тех же пяти запросах.

Математика — `books/02-sft-math.pdf`. Здесь все вызовы `peft` и `transformers` на виду.

In [ ]:
from common import MODEL_ID, SYSTEM, DATA, RUNS, demo_answers, show, side_by_side, policy_suite, fmt

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from vlmkit import ChatCollator, load_jsonl, memory_report, evaluate as ev
from vlmkit.compat import supported, first_accepted

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

## До

In [ ]:
suite = policy_suite()

before = demo_answers(model, processor, system=SYSTEM)
before_metrics = ev.run(model, processor, suite)

show(before, "ДО ОБУЧЕНИЯ")
print("\nметрики:", fmt(before_metrics))

## Адаптер

Три решения в конфиге, которые не косметика:

- **регекс с отрицательным просмотром** выкидывает визуальную башню — переучивать её на паре сотен примеров вредно;
- **`use_rslora=True`** меняет масштаб с `α/r` на `α/√r`, иначе вклад адаптера затухает с ростом ранга;
- **`r=16`** — для правил поведения достаточно, это изменение стиля, не новый навык.

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora)
model.print_trainable_parameters()

# Без этого при gradient checkpointing градиент не доходит до адаптеров.
# Ошибки не будет — обучение просто ничего не даст.
model.enable_input_require_grads()
model.config.use_cache = False

## Обучение

Чётные примеры учим, нечётные держим на замер — они в обучение не попадают.

`remove_unused_columns=False` критично: иначе `Trainer` выбросит из батча всё, чего нет в сигнатуре `forward`. `supported()` отсеивает аргументы, которых нет в вашей версии `transformers`.

In [ ]:
train_samples = load_jsonl(DATA / "policy.jsonl")[::2]
collator = ChatCollator(processor, system=SYSTEM)

args = dict(
    output_dir=str(RUNS / "sft-policy"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=1e-4,          # для LoRA в 100 раз выше полного FT — это нормально
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused",
    logging_steps=5,
    save_strategy="no",
    remove_unused_columns=False,
    report_to=[],
    seed=42,
)
args |= first_accepted(TrainingArguments, {"warmup_ratio": 0.05, "warmup_steps": 3})

trainer = Trainer(
    model=model,
    args=TrainingArguments(**supported(TrainingArguments, args)),
    train_dataset=train_samples,
    data_collator=collator,
)
result = trainer.train()
print(f"loss {result.training_loss:.3f}, {result.metrics.get('train_runtime', 0)/60:.1f} мин")

## После

Те же пять запросов. ✓ — модель вернула решение студенту (в ответе есть вопрос), · — выдала готовое. Правильная картина: ✓ на `clarify` и `guide`, · на `answer`.

In [ ]:
model.eval()
after = demo_answers(model, processor, system=SYSTEM)
after_metrics = ev.run(model, processor, suite)

show(after, "ПОСЛЕ ОБУЧЕНИЯ")
side_by_side(before, after, detector=lambda t: "?" in t)

print(f"\nдо:    {fmt(before_metrics)}")
print(f"после: {fmt(after_metrics)}")

## Адаптер как переключатель

Исходные веса не менялись. Адаптер можно выключить на лету — один процесс отвечает и с ограничениями, и без.

In [ ]:
with model.disable_adapter():
    print("адаптер выключен:", fmt(ev.run(model, processor, suite)))
print("адаптер включён: ", fmt(ev.run(model, processor, suite)))

model.save_pretrained(str(RUNS / "sft-policy"))

## На что смотреть

**Попадание выросло, ложные тоже выросли** — модель переобобщила: научилась уточнять и стала уточнять везде. Лечится больше примерами группы `answer` или меньшим числом эпох.

**Попадание не изменилось** — проверьте `preview` из `00-setup`: возможно, метки замаскированы целиком и градиента нет.

**Ответы стали обрывочными** — слишком высокая скорость обучения, попробуйте `5e-5`.